# 07 · References, Mutation & Functions as Values

Today has two big ideas. First: what really happens in memory when you copy an array or pass it to a function. It's one of the most important ideas in programming, and it works the same in almost every language. Second: functions are values too, which leads to `map`, `filter` and `reduce`.

**By the end of this lesson you can:**
- Explain the difference between copying a value and copying a reference
- Make real copies of arrays and objects
- Explain why a function can change an array you give it
- Pass functions to other functions (callbacks)
- Use `map`, `filter`, `reduce` and `sort`, and list comprehensions in Python

In [ ]:
# ▶️ Run this cell first, every time you open the notebook.
# It lets %%node cells run JavaScript and gives you check() in both languages.
%load_ext course_helpers

## 🔁 Warm-up
Answer from memory first: **no scrolling back, no notes**. Getting it wrong is fine. Trying to remember is what makes it stick.

1. How do you get the last item of an array in JavaScript? In Python?
2. What's the difference between `for...of` and `for...in` in JavaScript?
3. Write an object/dictionary describing a book with a title, an author and a year.
4. You changed a file on GitHub. Which command brings the change to your computer?

**My answers** (double-click to edit):

1. ✏️
2. ✏️
3. ✏️
4. ✏️

## Part 1: Values vs references

In [ ]:
%%node
// 🔮 My prediction:
let a = 10;
let b = a;
b = b + 1;
console.log("a:", a, " b:", b);

const list1 = [1, 2, 3];
const list2 = list1;
list2.push(4);
console.log("list1:", list1);
console.log("list2:", list2);

In [ ]:
# 🔮 My prediction:
a = 10
b = a
b = b + 1
print("a:", a, " b:", b)

list1 = [1, 2, 3]
list2 = list1
list2.append(4)
print("list1:", list1)
print("list2:", list2)

### 🔍 Investigate
Simple values (numbers, strings, booleans) are **copied**. Arrays and objects are **not** copied:
the variable holds a **reference**, an arrow pointing to where the array lives in memory.
`list2 = list1` copies the **arrow**, not the array.

```
a ──▶ 10              list1 ──┐
b ──▶ 11                      ├──▶ [1, 2, 3, 4]
                      list2 ──┘
```

Two names, **one** array. Change it through either name, and both "see" the change.
This works the same way in Python, Java, C# and most modern languages.
**Draw boxes and arrows whenever you're unsure.**

> Detail for the curious: in Python *every* variable is an arrow, but numbers and strings can never be
> changed (they're *immutable*), so they behave as if they were copied.

## Part 2: Making a real copy

In [ ]:
%%node
const original = [1, 2, 3];
const copy = [...original];      // "spread" the items into a NEW array
copy.push(4);
console.log(original, copy);

const person = { name: "Ada", age: 36 };
const personCopy = { ...person };
personCopy.age = 99;
console.log(person, personCopy);

In [ ]:
original = [1, 2, 3]
copy = original.copy()       # or list(original), or original[:]
copy.append(4)
print(original, copy)

person = {"name": "Ada", "age": 36}
person_copy = person.copy()
person_copy["age"] = 99
print(person, person_copy)

⚠️ These are **shallow** copies: only one level deep. If the array holds objects, the copy points to
the *same* objects. For a full (deep) copy use `structuredClone(x)` in JavaScript and
`copy.deepcopy(x)` in Python.

## Part 3: Functions and references

In [ ]:
%%node
// 🔮 My prediction:
function addItem(cart, item) {
  cart.push(item);
}

function resetCart(cart) {
  cart = [];
}

const myCart = ["apple"];
addItem(myCart, "bread");
console.log(myCart);

resetCart(myCart);
console.log(myCart);

### 🔍 Investigate
The parameter `cart` gets a copy of the **arrow**.
- `cart.push(item)` follows the arrow and changes the shared array, so `myCart` sees it.
- `cart = []` only turns the *local* arrow towards a new, empty array. `myCart` still points at the
  old one.

```
during addItem:                       during resetCart:
myCart ──┐                            myCart ──▶ ["apple", "bread"]
         ├──▶ ["apple", "bread"]
cart   ──┘                            cart   ──▶ []
```

Functions that change their inputs can surprise the caller. Functions that leave their inputs alone
and **return a new value** are called **pure** functions. They are easier to understand and test:

```js
function withItem(cart, item) {
  return [...cart, item];   // a new array, the original is untouched
}
```

## Part 4: When are two arrays "equal"?

In [ ]:
%%node
// 🔮 My prediction:
const x = [1, 2];
const y = [1, 2];
const z = x;

console.log(x === y);
console.log(x === z);
console.log(JSON.stringify(x) === JSON.stringify(y));

In [ ]:
# 🔮 My prediction:
x = [1, 2]
y = [1, 2]
z = x

print(x == y)
print(x is y)
print(x is z)

### 🔍 Investigate
- For arrays and objects, JavaScript's `===` asks **"is it the same array?"** (the same arrow), not
  "do they have the same contents?".
- Python has both: `==` compares **contents**, `is` compares **identity** (same object).
- Java works like JavaScript (`==` checks identity, `.equals()` compares contents).
- And now `const` makes sense: the arrow can't be moved, but the array it points to can still change.

## Part 5: Functions are values
A function is a value, just like a number or a string. You can store it in a variable and pass it to
another function.

In [ ]:
%%node
// 🔮 My prediction:
function shout(text) {
  return text.toUpperCase() + "!";
}

function whisper(text) {
  return text.toLowerCase() + "...";
}

function speak(text, style) {
  return style(text);
}

console.log(speak("Hello", shout));
console.log(speak("Hello", whisper));
console.log(speak("Hello", (t) => `🤖 ${t} 🤖`));

In [ ]:
def shout(text):
    return text.upper() + "!"

def whisper(text):
    return text.lower() + "..."

def speak(text, style):
    return style(text)

print(speak("Hello", shout))
print(speak("Hello", whisper))
print(speak("Hello", lambda t: f"🤖 {t} 🤖"))

A function passed into another function is called a **callback**. Notice: `shout` (no parentheses)
is the function itself; `shout("hi")` *calls* it.

## Part 6: `map`, `filter`, `reduce`
Three loop patterns are so common that arrays have built-in methods for them. Each one takes a callback.

| Method | What it does | Loop pattern |
|---|---|---|
| `map(f)` | a new array with `f(x)` for every item | transform |
| `filter(f)` | a new array with only the items where `f(x)` is true | select |
| `reduce(f, start)` | combines all items into one value | accumulator |

In [ ]:
%%node
const prices = [100, 250, 80, 400];

// The loop way
const withTaxLoop = [];
for (const p of prices) {
  withTaxLoop.push(p * 1.25);
}
console.log(withTaxLoop);

// The map way: same result
const withTax = prices.map((p) => p * 1.25);
console.log(withTax);

const cheap = prices.filter((p) => p < 200);
console.log(cheap);

const total = prices.reduce((sum, p) => sum + p, 0);
console.log(total);

console.log(prices);   // unchanged!

In [ ]:
prices = [100, 250, 80, 400]

with_tax = [p * 1.25 for p in prices]      # list comprehension = map
cheap = [p for p in prices if p < 200]      # ... with a filter
total = sum(prices)                         # the most common "reduce"

print(with_tax)
print(cheap)
print(total)

Python prefers **list comprehensions**: `[expression for item in list if condition]`. Python has
`map()` and `filter()` too, but they're used less.

You can chain the methods. Read it top to bottom like a recipe:

In [ ]:
%%node
const students = [
  { name: "Ada", grade: 92 },
  { name: "Linus", grade: 58 },
  { name: "Grace", grade: 85 },
  { name: "Alan", grade: 41 },
];

const passedNames = students
  .filter((s) => s.grade >= 60)
  .map((s) => s.name);

console.log(passedNames);

### Sorting

In [ ]:
%%node
// 🔮 My prediction:
const nums = [10, 9, 1, 100, 25];

console.log([...nums].sort());
console.log([...nums].sort((a, b) => a - b));

In [ ]:
nums = [10, 9, 1, 100, 25]
print(sorted(nums))

students = [{"name": "Ada", "grade": 92}, {"name": "Linus", "grade": 58}]
print(sorted(students, key=lambda s: s["grade"]))

### 🔍 Investigate
- JavaScript's `sort()` compares items **as text** by default, so `"10"` comes before `"9"`, like in a
  dictionary. For numbers, always give it a compare function: `(a, b) => a - b`. A negative result
  means "a comes first".
- `sort()` **changes the original array**. That's why we sorted a copy: `[...nums]`.
- Python's `sorted()` returns a new list and sorts numbers correctly. `key=` says what to sort by.

## 🛠️ Make
### 1. Grade report: no `for` loops allowed!

In [ ]:
%%node
const students = [
  { name: "Ada", grade: 92 },
  { name: "Linus", grade: 58 },
  { name: "Grace", grade: 85 },
  { name: "Alan", grade: 41 },
];

// ✏️ Use filter, map and reduce
const passed = [];     // the students with grade >= 60
const names = [];      // the names of ALL students
const average = 0;     // the average grade of all students

check(passed.length, 2);
check(names, ["Ada", "Linus", "Grace", "Alan"]);
check(average, 69);

### 2. A pure function

In [ ]:
%%node
function doubleAll(numbers) {
  // ✏️ Return a NEW array with every number doubled. Don't change the original!
}

const original = [1, 2, 3];
const doubled = doubleAll(original);
check(doubled, [2, 4, 6]);
check(original, [1, 2, 3]);   // must be unchanged

### 3. Write your own `map` and `filter`
There's no magic: they're just loops inside a function.

In [ ]:
%%node
function myMap(array, callback) {
  // ✏️ Build and return a new array with callback(item) for every item
}

function myFilter(array, test) {
  // ✏️ Build and return a new array with only the items where test(item) is true
}

check(myMap([1, 2, 3], (n) => n * 10), [10, 20, 30]);
check(myMap(["a", "b"], (s) => s.toUpperCase()), ["A", "B"]);
check(myFilter([1, 2, 3, 4], (n) => n % 2 === 0), [2, 4]);

### 4. Ranking

In [ ]:
%%node
const students = [
  { name: "Ada", grade: 92 },
  { name: "Linus", grade: 58 },
  { name: "Grace", grade: 85 },
  { name: "Alan", grade: 41 },
];

// ✏️ Make a sorted COPY, highest grade first. Don't change `students`!
const ranking = [];

check(ranking.map((s) => s.name), ["Ada", "Grace", "Linus", "Alan"]);
check(students[1].name, "Linus");   // the original order is unchanged

### 5. In Python, with list comprehensions and `sorted`

In [ ]:
students = [
    {"name": "Ada", "grade": 92},
    {"name": "Linus", "grade": 58},
    {"name": "Grace", "grade": 85},
    {"name": "Alan", "grade": 41},
]

# ✏️
passed = []     # the students with grade >= 60
names = []      # all names
ranking = []    # names, sorted by grade, best first (hint: reverse=True)

check(len(passed), 2)
check(names, ["Ada", "Linus", "Grace", "Alan"])
check(ranking, ["Ada", "Grace", "Linus", "Alan"])

## 💥 Break it

In [ ]:
%%node
// 🔮 My prediction:
const nums = [1, 2, 3];
const doubled = nums.map((n) => {
  n * 2;
});
console.log(doubled);

In [ ]:
# 🔮 My prediction:
grid = [[0, 0]] * 3
grid[0][0] = 5
print(grid)

### 🔍 Investigate
- An arrow function with `{ }` needs an explicit `return`. Without the braces (`(n) => n * 2`) the
  value is returned automatically.
- `[[0, 0]] * 3` makes a list with **three arrows to the same inner list**. Change one, and all three
  change. Draw it! The fix: `[[0, 0] for _ in range(3)]` makes three separate lists.

## 🌍 Rosetta Stone: same idea, different language
The ideas are the same in every language. Only the spelling changes. Add your own rows when you discover something new.

| Idea | JavaScript | Python |
|---|---|---|
| Copy an array / list | `[...a]` | `a.copy()` |
| Copy an object / dict | `{ ...o }` | `d.copy()` |
| Deep copy | `structuredClone(x)` | `copy.deepcopy(x)` |
| Same contents? | compare yourself (e.g. with JSON) | `a == b` |
| Same object? | `a === b` | `a is b` |
| Transform every item | `a.map((x) => x * 2)` | `[x * 2 for x in a]` |
| Keep some items | `a.filter((x) => x > 0)` | `[x for x in a if x > 0]` |
| Combine into one value | `a.reduce((s, x) => s + x, 0)` | `sum(a)` or a loop |
| Sort numbers (as a copy) | `[...a].sort((x, y) => x - y)` | `sorted(a)` |
| Sort by a property | `[...a].sort((x, y) => x.age - y.age)` | `sorted(a, key=lambda x: x["age"])` |

## ✍️ Explain it back
Answer in your own words, as if explaining it to a friend who missed today's session.

1. Explain, with a drawing, why changing `list2` also changed `list1`.
2. What's the difference between `map` and `filter`?
3. Why can a function that changes its input array cause bugs?

**My explanations** (double-click to edit):

1. ✏️
2. ✏️
3. ✏️

## 🌱 Git checkpoint
A **branch** is a separate line of work. `main` is your stable version. You try things on a branch
and **merge** them into `main` when you're happy. A branch is just a movable label that points at a commit.

```
main:        A ── B ── C ─────────── M      (M = merge)
                        \           /
lesson-07:               D ── E ───┘        (today's work)
```

```bash
git switch -c lesson-07          # create a new branch and move to it
# ... do the exercises, then:
git add 07_references_and_functions_as_values.ipynb
git commit -m "Complete lesson 07 exercises"

git switch main                  # 😮 reopen the notebook: your answers are gone!
git switch lesson-07             # ... and they're back
git switch main
git merge lesson-07              # bring the work into main
git push
git branch -d lesson-07          # delete the label (the commits stay)
git log --oneline --graph --all  # draw the history
```

> 💡 Switching branches changes the files on your disk. **Save before switching**, and reload the
> notebook afterwards (in VS Code: close it and open it again).